In [10]:
import sys, os, math, torch, time, random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.utils as U
import torch; 
from typing import Tuple
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Sampler

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [3]:
# загрузка данных
# путь к файлу
folder = "data"
trainFileName = "Si-12.25_2025-09-22_features.npy"
labelFileName = "Si-12.25_2025-09-22_prices.csv"

# загрузка фичей
X = np.load(os.path.join(folder, trainFileName))

print("Форма массива:", X.shape)
print("Тип данных:", X.dtype)

# загрузка меток
df_p = pd.read_csv(os.path.join(folder, labelFileName))
assert {"ms","mid"}.issubset(df_p.columns), "Нужны колонки ms и mid в prices.csv"
ms  = df_p["ms"].values.astype(np.int64)
mid = df_p["mid"].values.astype(np.float64)

assert len(X) == len(ms) == len(mid), f"Несовпадение длин: X={len(X)} ms={len(ms)} mid={len(mid)}"

Форма массива: (143799, 138)
Тип данных: float32


In [5]:
# ==== функции ====
def build_barrier_labels(ms: np.ndarray,
                         mid: np.ndarray,
                         tick_size: float = 1.0,
                         theta_ticks: int = 1,
                         horizon_sec: float = 1.5) -> np.ndarray:
    """
    Triple-barrier по реальному времени t -> t+τ:
      класс 2: цена поднималась >= +θ,
      класс 0: цена падала  <= -θ,
      класс 1: ни то ни другое (flat).
    Там, где нет будущего окна, возвращаем -1.
    """
    N = len(mid)
    y = np.full(N, -1, dtype=np.int64)
    theta = theta_ticks * tick_size

    import bisect
    for i in range(N):
        t0 = ms[i]
        t_end = t0 + int(horizon_sec * 1000)
        j = bisect.bisect_right(ms, t_end, lo=i+1)
        if j <= i+1:
            continue
        m0 = mid[i]
        w = mid[i+1:j]
        if w.size == 0:
            continue
        up_hit = (w.max() - m0) >= theta
        dn_hit = (w.min() - m0) <= -theta
        if up_hit and not dn_hit:
            y[i] = 2
        elif dn_hit and not up_hit:
            y[i] = 0
        elif up_hit and dn_hit:
            # кто наступил раньше (грубо)
            up_idx = np.argmax(w == w.max())
            dn_idx = np.argmax(w == w.min())
            y[i] = 2 if up_idx < dn_idx else 0
        else:
            y[i] = 1
    return y

def make_windows(X2D: np.ndarray, T: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Скользящие окна по времени (каузально):
      вход: X2D (N, F)
      выход: Xwin (N-T+1, T, F), end_idx — индексы последних точек окон
    """
    N, F = X2D.shape
    if N < T:
        raise ValueError(f"Мало данных для окна: N={N} < T={T}")
    s0, s1 = X2D.strides
    Xwin = np.lib.stride_tricks.as_strided(
        X2D, shape=(N - T + 1, T, F), strides=(s0, s0, s1)
    ).copy()
    end_idx = np.arange(T-1, N)
    return Xwin, end_idx

In [6]:
# ==== параметры задачи ====
TICK_SIZE   = 1.0   # тик инструмента
THETA_TICKS = 5     # порог в тиках (±1)
HORIZON_SEC = 2     # горизонт (сек)
T           = 240    # длина окна (шагов), умнодить на время между срезами (0.3сек), полученное значение должно быть в 10-40 раз больше HORIZON_SEC

# ==== построение меток и окон ====
y_all = build_barrier_labels(ms, mid, tick_size=TICK_SIZE,
                             theta_ticks=THETA_TICKS, horizon_sec=HORIZON_SEC)

valid = (y_all != -1)
Xv, yv, msv, midv = X[valid], y_all[valid], ms[valid], mid[valid]

Xwin, end_idx = make_windows(Xv, T)
y_win  = yv[end_idx]
ms_win = msv[end_idx]

print(f"Классы и счётчики: {np.unique(y_win, return_counts=True)}")
print("Окна:", Xwin.shape, "| метки:", y_win.shape)

# ==== временной сплит: train / val / test ====
Nw = len(Xwin)
i_tr = int(Nw * 0.70)
i_va = int(Nw * 0.85)

X_train, y_train = Xwin[:i_tr], y_win[:i_tr]
X_val,   y_val   = Xwin[i_tr:i_va], y_win[i_tr:i_va]
X_test,  y_test  = Xwin[i_va:],     y_win[i_va:]

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Классы и счётчики: (array([0, 1, 2], dtype=int64), array([  6596, 128575,   6694], dtype=int64))
Окна: (141865, 240, 138) | метки: (141865,)
Train: (99305, 240, 138) Val: (21280, 240, 138) Test: (21280, 240, 138)


In [11]:
# -------- Dataset --------
class NpWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.astype(np.float32, copy=False)
        self.y = y.astype(np.int64,  copy=False)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i].T.copy())  # (F, T) для Conv1d
        y = torch.tensor(self.y[i])
        return x, y

train_ds = NpWindowDataset(X_train, y_train)
val_ds   = NpWindowDataset(X_val,   y_val)
test_ds  = NpWindowDataset(X_test,  y_test)

# -------- Balanced Batch Sampler --------
class BalancedBatchSampler(Sampler):
    """
    Формирует батчи одинакового размера из равного числа классов (по возможности).
    Например, при batch_size=512 и 3 классах -> по ~170 сэмплов с класса.
    Для редких классов крутит циклически (replacement=True).
    """
    def __init__(self, labels, batch_size=512, num_classes=3, seed=42):
        self.labels = np.asarray(labels)
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.rng = random.Random(seed)

        self.idx_by_cls = [np.flatnonzero(self.labels == c).tolist() for c in range(num_classes)]
        for lst in self.idx_by_cls:
            self.rng.shuffle(lst)

        self.ptr = [0]*num_classes
        self.len_dataset = len(self.labels)
        # сколько батчей сделаем за эпоху (ориентируемся на «средний» объём)
        self.batches_per_epoch = math.floor(self.len_dataset / self.batch_size)

    def __len__(self):
        return self.batches_per_epoch

    def __iter__(self):
        per_cls = [self.batch_size // self.num_classes] * self.num_classes
        rem = self.batch_size - sum(per_cls)
        # раздадим остаток по первым классам
        for c in range(rem):
            per_cls[c] += 1

        for _ in range(self.batches_per_epoch):
            batch_idx = []
            for c in range(self.num_classes):
                need = per_cls[c]
                lst = self.idx_by_cls[c]
                # если не хватает — крутим по кругу
                if self.ptr[c] + need > len(lst):
                    # перетасуем чтобы не циклилось одинаково
                    self.rng.shuffle(lst)
                    self.ptr[c] = 0
                batch_idx.extend(lst[self.ptr[c]: self.ptr[c]+need])
                self.ptr[c] += need
            self.rng.shuffle(batch_idx)
            yield from batch_idx  # DataLoader сам нарежет по batch_size

# ВАЖНО: говорим DataLoader'у, что уже получили «индексы подряд»
class BalancedBatchBatchSampler(Sampler):
    """Обёртка: превращает поток индексов из BalancedBatchSampler в батчи по batch_size."""
    def __init__(self, base_sampler, batch_size):
        self.base = base_sampler
        self.batch_size = batch_size
    def __len__(self):
        return len(self.base)
    def __iter__(self):
        it = iter(self.base)
        for _ in range(len(self.base)):
            batch = [next(it) for __ in range(self.batch_size)]
            yield batch

batch_size = 512
base_sampler = BalancedBatchSampler(y_train, batch_size=batch_size, num_classes=3, seed=42)
batch_sampler = BalancedBatchBatchSampler(base_sampler, batch_size)

train_dl = DataLoader(train_ds, batch_sampler=batch_sampler)
val_dl   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_dl  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [16]:
# -------- Модель (GroupNorm + Dropout остались) --------
class TemporalConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=5, d=1, groups=8):
        super().__init__()
        pad = (k - 1) * d
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=k, dilation=d, padding=pad)
        self.act1  = nn.GELU()
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, dilation=2, padding=2*2)
        self.act2  = nn.GELU()
        g = min(groups, out_ch)
        self.norm  = nn.GroupNorm(num_groups=g, num_channels=out_ch)
    def forward(self, x):
        y = self.conv1(x); y = self.act1(y)
        y = self.conv2(y); y = self.act2(y)
        y = self.norm(y)
        return y[..., :x.shape[-1]]

class DeepLOBLike(nn.Module):
    def __init__(self, F, num_classes=3, hidden=128, groups=8, p_drop=0.3):
        super().__init__()
        self.stem = nn.Conv1d(F, hidden, kernel_size=1)
        self.b1 = TemporalConvBlock(hidden, hidden, k=5, d=1, groups=groups)
        self.b2 = TemporalConvBlock(hidden, hidden, k=5, d=2, groups=groups)
        self.b3 = TemporalConvBlock(hidden, hidden, k=5, d=4, groups=groups)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.b1(x); x = self.b2(x); x = self.b3(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)

_, T, F = X_train.shape
model = DeepLOBLike(F=F, num_classes=3, hidden=128).to(device)

# -------- Умеренный Focal Loss --------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = None if alpha is None else torch.tensor(alpha, dtype=torch.float32)
        self.gamma = gamma
        self.reduction = reduction
    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(
            logits, targets, reduction='none',
            weight=(self.alpha.to(logits.device) if self.alpha is not None else None)
        )
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction=="mean" else loss.sum()

criterion = FocalLoss(alpha=[3.0, 1.0, 3.0], gamma=2.0)  # не давим flat в ноль

# -------- Оптимизатор/шедулер/AMP/клип --------
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=3e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

In [17]:
def evaluate(dl):
    model.eval()
    total, n = 0.0, 0
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            # мониторим «обычный» CE (он коррелирует со здравым смыслом)
            loss = nn.functional.cross_entropy(logits, yb, reduction='mean')
            total += loss.item() * xb.size(0); n += xb.size(0)
            y_true.append(yb.cpu().numpy())
            y_pred.append(logits.argmax(1).cpu().numpy())
    y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred)
    val_loss = total / max(1,n)
    macroF1 = f1_score(y_true, y_pred, average="macro")
    f1d = f1_score(y_true, y_pred, labels=[0], average=None)[0]
    f1f = f1_score(y_true, y_pred, labels=[1], average=None)[0]
    f1u = f1_score(y_true, y_pred, labels=[2], average=None)[0]
    cm  = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    return val_loss, macroF1, (f1d, f1f, f1u), cm

best_score, best_state = -1.0, None
epochs = 15

for ep in range(1, epochs+1):
    model.train()
    total, correct, n = 0.0, 0, 0
    tic = time.time()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        U.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        total += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        n += xb.size(0)

    train_loss = total / n
    train_acc  = correct / n
    scheduler.step()

    val_loss, macroF1, (f1d,f1f,f1u), cm = evaluate(val_dl)
    print(f"[{ep:02d}] train_loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} macroF1={macroF1:.3f} "
          f"F1↓={f1d:.3f} F1○={f1f:.3f} F1↑={f1u:.3f}  ({time.time()-tic:.1f}s)")
    if ep in (1, 5, 10, 15):
        print("Confusion matrix [rows=true, cols=pred]:\n", cm)

    if macroF1 > best_score:
        best_score = macroF1
        best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}


[01] train_loss=1.9264 acc=0.335 | val_loss=1.7996 macroF1=0.043 F1↓=0.000 F1○=0.000 F1↑=0.129  (26.9s)
Confusion matrix [rows=true, cols=pred]:
 [[    0     0  1433]
 [    0     0 18383]
 [    0     0  1464]]
[02] train_loss=1.9155 acc=0.338 | val_loss=1.7906 macroF1=0.087 F1↓=0.133 F1○=0.000 F1↑=0.128  (26.5s)
[03] train_loss=1.7419 acc=0.435 | val_loss=1.7198 macroF1=0.101 F1↓=0.151 F1○=0.000 F1↑=0.153  (26.5s)
[04] train_loss=1.5259 acc=0.493 | val_loss=1.6507 macroF1=0.102 F1↓=0.146 F1○=0.014 F1↑=0.148  (26.7s)
[05] train_loss=1.2949 acc=0.545 | val_loss=1.6575 macroF1=0.100 F1↓=0.143 F1○=0.015 F1↑=0.143  (26.7s)
Confusion matrix [rows=true, cols=pred]:
 [[ 797    3  633]
 [8406  142 9835]
 [ 545    1  918]]
[06] train_loss=0.9823 acc=0.604 | val_loss=1.5474 macroF1=0.111 F1↓=0.133 F1○=0.055 F1↑=0.146  (26.3s)
[07] train_loss=0.7607 acc=0.645 | val_loss=1.6528 macroF1=0.109 F1↓=0.140 F1○=0.049 F1↑=0.137  (26.8s)
[08] train_loss=0.6267 acc=0.673 | val_loss=1.5658 macroF1=0.129 F1↓=

In [ ]:
# тест
if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), "best_deeplob_like.pt")

# отчёт на тесте
model.eval()
all_p, all_y = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device)
        logits = model(xb)
        all_p.append(logits.argmax(1).cpu().numpy())
        all_y.append(yb.numpy())
y_pred = np.concatenate(all_p); y_true = np.concatenate(all_y)
print("\nЛучший macro-F1 (val):", round(best_score, 4))
print(classification_report(y_true, y_pred, digits=3, target_names=["down","flat","up"]))
print("Сохранено:", os.path.abspath("best_deeplob_like.pt"))